# byteSmart Basic ML Practice

This notebook keeps the machine learning examples simple and focused on the three algorithms requested: **linear regression**, **K-Means**, and **logistic regression**.

The goal is not to make the most complex model. The goal is to understand the dataset, make clear graphs, and explain what the models show about dry ice loss and temperature behavior.

## 1. Setup

This cell imports the main Python libraries and mounts Google Drive so the notebook can find the dataset zip file.

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.cluster import KMeans
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import ConfusionMatrixDisplay, classification_report
    from sklearn.preprocessing import StandardScaler
except ImportError:
    %pip -q install scikit-learn
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.cluster import KMeans
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import ConfusionMatrixDisplay, classification_report
    from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive mount skipped:', e)

## 2. Load the Data

The dataset is stored as a zip file. This code searches Google Drive and the Colab files area for the zip, extracts it, and loads the CSV files.

In [ ]:
ZIP_NAME = '14888121-20260708T173347Z-3-001.zip'

search_places = [Path('/content'), Path('/content/drive/MyDrive')]
zip_path = None

for place in search_places:
    if place.exists():
        matches = list(place.rglob(ZIP_NAME))
        if matches:
            zip_path = matches[0]
            break

if zip_path is None:
    raise FileNotFoundError('Upload the dataset zip file to Colab or Google Drive first.')

extract_dir = Path('/content/byteSmart_basic_data')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

data_dir = extract_dir / '14888121'
print('Dataset loaded from:', data_dir)
print([p.name for p in data_dir.glob('*.csv')])

## 3. Clean the Data

The CSV files have extra header rows, so this cell removes those rows and converts the useful columns into numbers.

In [ ]:
def elapsed_to_hours(series):
    return pd.to_timedelta(series.astype(str), errors='coerce').dt.total_seconds() / 3600

def load_dry_ice(path):
    raw = pd.read_csv(path, header=None)
    dry = pd.DataFrame({
        'hours': pd.to_numeric(raw.iloc[4:, 2], errors='coerce'),
        'baseline_lb': pd.to_numeric(raw.iloc[4:, 5], errors='coerce'),
        'refrigerated_lb': pd.to_numeric(raw.iloc[4:, 6], errors='coerce'),
    })
    return dry.dropna(subset=['hours'])

def load_test1(path):
    df = pd.read_csv(path, low_memory=False).iloc[2:].copy()
    df['hours'] = elapsed_to_hours(df['Time Elapsed'])
    for col in df.columns:
        if col not in ['date', 'time', 'Time Elapsed', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['hours'])

def load_test2(path):
    df = pd.read_csv(path, low_memory=False).iloc[1:].copy()
    df['timestamp'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    df['hours'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 3600
    for col in df.columns:
        if col not in ['TIMESTAMP', 'timestamp', 'hours']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['hours'])

def get_temp_cols(df, exclude):
    cols = []
    for col in df.columns:
        if col not in exclude and pd.api.types.is_numeric_dtype(df[col]) and df[col].notna().sum() > 100:
            cols.append(col)
    return cols

dry = load_dry_ice(data_dir / 'Test1_DryIceWeight.csv')
test1 = load_test1(data_dir / 'Test1_TempCO2O2.csv')
test2 = load_test2(data_dir / 'Test2_TempCO2O2.csv')

test1_temp_cols = get_temp_cols(test1, {'date', 'time', 'Time Elapsed', 'hours', 'O2', 'CO2', 'Ambient', 'Unnamed: 63', 'Unnamed: 64'})
test2_temp_cols = get_temp_cols(test2, {'TIMESTAMP', 'timestamp', 'hours', 'O2', 'CO2'})

print('Dry ice rows:', len(dry))
print('Test 1 temperature sensors:', len(test1_temp_cols))
print('Test 2 temperature sensors:', len(test2_temp_cols))

## 4. Linear Regression: Dry Ice Mass Drop

Linear regression draws a simple best-fit line. Here, the slope tells us how many pounds of dry ice are lost per hour.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
summary = []

for col, label, color in [
    ('baseline_lb', 'Baseline', 'crimson'),
    ('refrigerated_lb', 'Refrigerated', 'royalblue')
]:
    data = dry[['hours', col]].dropna()
    model = LinearRegression()
    model.fit(data[['hours']], data[col])

    slope = model.coef_[0]
    summary.append({
        'condition': label,
        'loss_rate_lb_per_hour': slope,
        'loss_rate_lb_per_day': slope * 24,
    })

    ax.scatter(data['hours'], data[col], label=f'{label} data', color=color, alpha=0.7)
    ax.plot(data['hours'], model.predict(data[['hours']]), color=color, linewidth=2, label=f'{label} regression line')

ax.set_title('Linear Regression: Dry Ice Mass Over Time')
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('Dry ice mass (lb)')
ax.legend()
plt.show()

pd.DataFrame(summary)

**Explanation:** The baseline dry ice line drops faster than the refrigerated line. This means the baseline condition lost dry ice more quickly, while the refrigerated condition preserved the dry ice longer.

## 5. Linear Regression: Fastest-Warming Test 1 Sensors

This uses one simple linear regression line per sensor. The sensors with the biggest positive slopes warmed the fastest.

In [ ]:
sensor_rates = []

for sensor in test1_temp_cols:
    data = test1[['hours', sensor]].dropna()
    if len(data) > 2:
        model = LinearRegression()
        model.fit(data[['hours']], data[sensor])
        sensor_rates.append({
            'sensor': sensor,
            'warming_rate_F_per_hour': model.coef_[0]
        })

sensor_rates = pd.DataFrame(sensor_rates).sort_values('warming_rate_F_per_hour', ascending=False)
top_warming = sensor_rates.head(10)

top_warming.plot(
    kind='barh',
    x='sensor',
    y='warming_rate_F_per_hour',
    figsize=(9, 5),
    color='seagreen',
    legend=False
)
plt.title('Test 1 Sensors That Warmed the Fastest')
plt.xlabel('Warming rate (degrees F per hour)')
plt.ylabel('Sensor')
plt.gca().invert_yaxis()
plt.show()

top_warming

**Explanation:** The tallest bars show the sensors that warmed fastest in Test 1. These sensors are important because they show the locations where the cold-chain setup was weakest.

## 6. Test 1 Sensor Spread Over Time

This graph shows the difference between the warmest and coldest Test 1 sensor at each time. A bigger spread means the temperature was more uneven across the container.

In [ ]:
test1_spread = test1[test1_temp_cols].max(axis=1) - test1[test1_temp_cols].min(axis=1)

plt.figure(figsize=(10, 5))
plt.plot(test1['hours'], test1_spread, color='mediumpurple')
plt.title('Test 1 Temperature Spread Over Time')
plt.xlabel('Elapsed time (hours)')
plt.ylabel('Warmest sensor minus coldest sensor (degrees F)')
plt.show()

pd.DataFrame({
    'average_spread_F': [test1_spread.mean()],
    'median_spread_F': [test1_spread.median()],
    'maximum_spread_F': [test1_spread.max()]
})

**Explanation:** The spread is large for most of the test, meaning Test 1 had uneven temperature conditions. The spikes show moments when one location became much warmer than another.

## 7. K-Means: Group Similar Sensors

K-Means is used to group sensors with similar temperature behavior. This is a simple way to summarize many sensors at once.

In [ ]:
sensor_summary = pd.DataFrame({
    'sensor': test1_temp_cols,
    'average_temp_F': [test1[c].mean() for c in test1_temp_cols],
    'temp_std_F': [test1[c].std() for c in test1_temp_cols],
    'minimum_temp_F': [test1[c].min() for c in test1_temp_cols],
    'maximum_temp_F': [test1[c].max() for c in test1_temp_cols],
})

features = ['average_temp_F', 'temp_std_F', 'minimum_temp_F', 'maximum_temp_F']
X = sensor_summary[features]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
sensor_summary['cluster'] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
for cluster in sorted(sensor_summary['cluster'].unique()):
    group = sensor_summary[sensor_summary['cluster'] == cluster]
    plt.scatter(group['average_temp_F'], group['temp_std_F'], label=f'Cluster {cluster}', s=70)

plt.title('K-Means: Test 1 Sensor Groups')
plt.xlabel('Average temperature (degrees F)')
plt.ylabel('Temperature variation / standard deviation')
plt.legend()
plt.show()

sensor_summary.groupby('cluster')[features].mean().round(2)

**Explanation:** Each dot is a sensor. Sensors near each other behaved similarly. K-Means grouped the sensors into three basic groups, which makes the large sensor dataset easier to understand.

## 8. Logistic Regression: Categorize Test 1 vs Test 2

Logistic regression is used here as a simple classifier. It tries to decide whether a small time window looks like Test 1 or Test 2 based on temperature spread, average temperature, O2, and CO2.

In [ ]:
def make_simple_windows(df, temp_cols, label, window_size=120):
    rows = []
    simple = pd.DataFrame({
        'hours': df['hours'],
        'mean_temp_F': df[temp_cols].mean(axis=1),
        'spread_F': df[temp_cols].max(axis=1) - df[temp_cols].min(axis=1),
        'O2': df['O2'],
        'CO2': df['CO2'],
    }).dropna()

    for start in range(0, len(simple) - window_size, window_size):
        window = simple.iloc[start:start + window_size]
        rows.append({
            'test_label': label,
            'mean_temp_F': window['mean_temp_F'].mean(),
            'spread_F': window['spread_F'].mean(),
            'O2_mean': window['O2'].mean(),
            'CO2_mean': window['CO2'].mean(),
        })
    return pd.DataFrame(rows)

windows = pd.concat([
    make_simple_windows(test1, test1_temp_cols, 'Test 1'),
    make_simple_windows(test2, test2_temp_cols, 'Test 2')
], ignore_index=True)

X = windows[['mean_temp_F', 'spread_F', 'O2_mean', 'CO2_mean']]
y = windows['test_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
predictions = log_model.predict(X_test)

print(classification_report(y_test, predictions))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=['Test 1', 'Test 2'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Logistic Regression: Test 1 vs Test 2')
ax.set_xlabel('Predicted test')
ax.set_ylabel('Actual test')
plt.show()

**Explanation:** The confusion matrix shows how often the model guessed the correct test. If most values are on the diagonal, the model did well. In this case, logistic regression can categorize Test 1 and Test 2 well because the tests have different temperature and gas patterns.

## Final Summary

- Linear regression showed that dry ice mass dropped faster in the baseline condition than in the refrigerated condition.
- Linear regression also identified the Test 1 sensors that warmed the fastest.
- The Test 1 spread graph showed that temperatures were uneven across the container.
- K-Means grouped similar sensors together so the large dataset was easier to understand.
- Logistic regression categorized whether data looked like Test 1 or Test 2 and showed that the two tests had different patterns.